In [ ]:
import math
from array import array
import ROOT
import datetime
import pythia8 

In [ ]:
date_time=datetime.datetime.now()
time_seed=date_time.time().strftime('%H%M%S')
print(time_seed)


115239


In [ ]:
MUON_MASS_GEV = 0.10566
PION_MASS_GEV = 0.139
LUMI_FB = 110.0

# Part 1.

In [ ]:
def setup_signal(pythia):
    pythia.readString("Beams:idA = 2212")
    pythia.readString("Beams:idB = 2212")
    pythia.readString("Beams:eCM = 13600.")

    # Higgs production (simplest)
    pythia.readString("HiggsSM:all = on") #use all higgs channels

    # Force H -> ZZ and Z -> mu mu
    pythia.readString("25:onMode = off")
    pythia.readString("25:onIfMatch = 23 23")

    pythia.readString("23:onMode = off")
    pythia.readString("23:onIfMatch = 13 -13")

    # More stable generation
    pythia.readString("Random:setSeed = on")
    pythia.readString(f"Random:seed = {time_seed}")

In [ ]:

def setup_background(pythia):
    pythia.readString("Beams:idA = 2212")
    pythia.readString("Beams:idB = 2212")
    pythia.readString("Beams:eCM = 13600.")

    pythia.readString("WeakDoubleBoson:ffbar2gmZgmZ = on")
    pythia.readString("WeakZ0:gmZmode = 0") #mode 0 is the default

    pythia.readString("23:onMode = off")
    pythia.readString("23:onIfMatch = 13 -13")

    pythia.readString("Random:setSeed = on")
    pythia.readString(f"Random:seed = {time_seed}") 
    

In [ ]:
def pass_trigger(mu_pts, mu_etas):
    
    good = 0
    for pt, eta in zip(mu_pts, mu_etas):
        if pt > 20.0 and abs(eta) < 2.1:
            good += 1
        if good >= 2:
            return True
    return False


In [ ]:

def run(sample, n_events, out_file):
    pythia = pythia8.Pythia()

    if sample == "signal":
        setup_signal(pythia)
    else:
        setup_background(pythia)

    pythia.init()
    # print([x for x in dir(pythia) if 'sigma' in x.lower() or 'info' in x.lower()])
    
    # --- ROOT output ---
    fout = ROOT.TFile(out_file, "RECREATE")
    tree = ROOT.TTree("Events", "Generator-level events passing trigger")

    runTag = ROOT.std.string(sample)
    nMu = array('i', [0])
    trig = array('i', [0])
    weight = array('f', [1.0])

    mu_pt = ROOT.std.vector('float')()
    mu_eta = ROOT.std.vector('float')()
    mu_phi = ROOT.std.vector('float')()
    mu_q   = ROOT.std.vector('int')()

    pi_pt = ROOT.std.vector('float')()
    pi_eta = ROOT.std.vector('float')()
    pi_phi = ROOT.std.vector('float')()
    pi_q = ROOT.std.vector('int')()

    tree.Branch("runTag", runTag)
    tree.Branch("nMu", nMu, "nMu/I")
    tree.Branch("trig", trig, "trig/I")
    tree.Branch("weight", weight, "weight/F")
    tree.Branch("mu_pt", mu_pt)
    tree.Branch("mu_eta", mu_eta)
    tree.Branch("mu_phi", mu_phi)
    tree.Branch("mu_q", mu_q)
    tree.Branch("pi_pt", pi_pt)
    tree.Branch("pi_eta", pi_eta)
    tree.Branch("pi_phi", pi_phi)
    tree.Branch("pi_q", pi_q)

    # counters for efficiency
    n_total = 0
    n_pass  = 0

    for i in range(n_events):
        if not pythia.next():
            continue

        n_total += 1


        mu_pt.clear(); mu_eta.clear(); mu_phi.clear(); mu_q.clear()
        pi_pt.clear(); pi_eta.clear(); pi_phi.clear(); pi_q.clear()

        for p in pythia.event:
            if not p.isFinal():
                continue

            if abs(p.id()) == 13:
                mu_pt.push_back(p.pT())
                mu_eta.push_back(p.eta())
                mu_phi.push_back(p.phi())
                mu_q.push_back(1 if p.id() == -13 else -1)

            if abs(p.id()) == 211:
                pi_pt.push_back(p.pT())
                pi_eta.push_back(p.eta())
                pi_phi.push_back(p.phi())
                pi_q.push_back(1 if p.id() == 211 else -1)

        nMu[0] = int(mu_pt.size())

        mu_pts  = [mu_pt[j] for j in range(mu_pt.size())]
        mu_etas = [mu_eta[j] for j in range(mu_eta.size())]
        fired = pass_trigger(mu_pts, mu_etas)
        trig[0] = 1 if fired else 0

        if fired:
            n_pass += 1
            tree.Fill()

    meta = ROOT.TTree("Meta", "Counters and cross section info")

    Ntotal = array('l', [n_total])
    Npass  = array('l', [n_pass])


    sigma_fb = array('d', [pythia.infoPython().sigmaGen() * 1.0e12])
    sigmaerr_fb = array('d', [pythia.infoPython().sigmaErr() * 1.0e12])

    

    meta.Branch("Ntotal", Ntotal, "Ntotal/L")
    meta.Branch("Npass",  Npass,  "Npass/L")    
    meta.Branch("SigmaFb", sigma_fb, "SigmaFb/D")
    meta.Branch("SigmaErrFb", sigmaerr_fb, "SigmaErrFb/D")


    meta.Fill()

    fout.Write()
    fout.Close()

    if n_total > 0:
        eff = n_pass / n_total
        err = math.sqrt(eff * (1 - eff) / n_total)
        print(f"[{sample}] Ntotal={n_total}  Npass={n_pass}")
        print(f"[{sample}] Trigger efficiency = {eff:.4f} ± {err:.4f}")


In [ ]:
total_event_cnt=1e6
sig_bkg_ratio=0.9469378353915646
sig_ratio=sig_bkg_ratio
bkg_ratio=1-sig_bkg_ratio

1000000.0


In [ ]:
run(sample="signal", n_events=int(total_event_cnt*sig_ratio), out_file="signal.root")
run(sample="background", n_events=int(total_event_cnt*bkg_ratio), out_file="background.root")

[signal] Ntotal=946937  Npass=616564
[signal] Trigger efficiency = 0.6511 ± 0.0005
[background] Ntotal=53062  Npass=33119
[background] Trigger efficiency = 0.6242 ± 0.0021

 *------------------------------------------------------------------------------------* 
 |                                                                                    | 
 |  *------------------------------------------------------------------------------*  | 
 |  |                                                                              |  | 
 |  |                                                                              |  | 
 |  |   PPP   Y   Y  TTTTT  H   H  III    A      Welcome to the Lund Monte Carlo!  |  | 
 |  |   P  P   Y Y     T    H   H   I    A A     This is PYTHIA version 8.317      |  | 
 |  |   PPP     Y      T    HHHHH   I   AAAAA    Last date of change: 20 Jan 2026  |  | 
 |  |   P       Y      T    H   H   I   A   A                                      |  | 
 |  |   P       Y      T  